# Introduction to DataFrames


In [1]:
import polars as pl
path_data = '../../../assets/data/'
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

cones = pl.read_csv(path_data + 'cones.csv')
nba = pl.read_csv(path_data + 'nba_salaries.csv').rename({"'15-'16 SALARY": "SALARY"})
# POLARS_CHANGE: movies_by_year.csv separates rows with carriage returns
movies = pl.read_csv(path_data + 'movies_by_year.csv', eol_char='\r')

We can now apply Python to analyze data. We will work with data stored in DataFrame structures.

DataFrames are a fundamental way of representing data sets. A DataFrame can be viewed in two ways:
* a sequence of named columns that each describe a single attribute of all entries in a data set, or
* a sequence of rows that each contain all information about a single individual in a data set.

We will study DataFrames in great detail in the next several chapters. For now, we will just introduce a few methods without going into technical details. 

The DataFrame `cones` has been imported for us; later we will see how, but here we will just work with it. First, let's take a look at it.

In [2]:
cones

Flavor,Color,Price
str,str,f64
"""strawberry""","""pink""",3.55
"""chocolate""","""light brown""",4.75
"""chocolate""","""dark brown""",5.25
"""strawberry""","""pink""",5.25
"""chocolate""","""dark brown""",5.25
"""bubblegum""","""pink""",4.75


The DataFrame has six rows. Each row corresponds to one ice cream cone. The ice cream cones are the *individuals*.

Each cone has three attributes: flavor, color, and price. Each column contains the data on one of these attributes, and so all the entries of any single column are of the same kind. Each column has a label. We will refer to columns by their labels.

A DataFrame method is just like a function, but it must operate on a DataFrame. So the call looks like

`name_of_dataframe.method(arguments)`

For example, if you want to see just the first two rows of a DataFrame, you can use the DataFrame method `head`.

In [3]:
cones.head(2)

Flavor,Color,Price
str,str,f64
"""strawberry""","""pink""",3.55
"""chocolate""","""light brown""",4.75


You can replace 2 by any number of rows. If you ask for more than six, you will only get six, because `cones` only has six rows.

## Choosing Sets of Columns
The method `select` creates a new DataFrame consisting of only the specified columns.

In [4]:
cones.select('Flavor')

Flavor
str
"""strawberry"""
"""chocolate"""
"""chocolate"""
"""strawberry"""
"""chocolate"""
"""bubblegum"""


This leaves the original DataFrame unchanged.

In [5]:
cones

Flavor,Color,Price
str,str,f64
"""strawberry""","""pink""",3.55
"""chocolate""","""light brown""",4.75
"""chocolate""","""dark brown""",5.25
"""strawberry""","""pink""",5.25
"""chocolate""","""dark brown""",5.25
"""bubblegum""","""pink""",4.75


You can select more than one column, by separating the column labels by commas.

In [6]:
cones.select('Flavor', 'Price')

Flavor,Price
str,f64
"""strawberry""",3.55
"""chocolate""",4.75
"""chocolate""",5.25
"""strawberry""",5.25
"""chocolate""",5.25
"""bubblegum""",4.75


You can also *drop* columns you don't want. The DataFrame above can be created by dropping the `Color` column.

In [7]:
cones.drop('Color')

Flavor,Price
str,f64
"""strawberry""",3.55
"""chocolate""",4.75
"""chocolate""",5.25
"""strawberry""",5.25
"""chocolate""",5.25
"""bubblegum""",4.75


You can name this new DataFrame and look at it again by just typing its name.

In [8]:
no_colors = cones.drop('Color')

no_colors

Flavor,Price
str,f64
"""strawberry""",3.55
"""chocolate""",4.75
"""chocolate""",5.25
"""strawberry""",5.25
"""chocolate""",5.25
"""bubblegum""",4.75


Like `select`, the `drop` method creates a smaller DataFrame and leaves the original DataFrame unchanged. In order to explore your data, you can create any number of smaller DataFrames by using choosing or dropping columns. It will do no harm to your original DataFrame.

## Sorting Rows

The `sort` method creates a new DataFrame by arranging the rows of the original DataFrame in ascending order of the values in the specified column. Here the `cones` DataFrame has been sorted in ascending order of the price of the cones.

In [9]:
cones.sort('Price')

Flavor,Color,Price
str,str,f64
"""strawberry""","""pink""",3.55
"""chocolate""","""light brown""",4.75
"""bubblegum""","""pink""",4.75
"""chocolate""","""dark brown""",5.25
"""strawberry""","""pink""",5.25
"""chocolate""","""dark brown""",5.25


To sort in descending order, you can use an *optional* argument to `sort`. As the name implies, optional arguments don't have to be used, but they can be used if you want to change the default behavior of a method. 

By default, `sort` sorts in increasing order of the values in the specified column. To sort in decreasing order, use the optional argument `descending=True`.

In [10]:
cones.sort('Price', descending=True)

Flavor,Color,Price
str,str,f64
"""chocolate""","""dark brown""",5.25
"""strawberry""","""pink""",5.25
"""chocolate""","""dark brown""",5.25
"""chocolate""","""light brown""",4.75
"""bubblegum""","""pink""",4.75
"""strawberry""","""pink""",3.55


Like `select` and `drop`, the `sort` method leaves the original DataFrame unchanged.

## Selecting Rows that Satisfy a Condition
The `filter` method creates a new DataFrame consisting only of the rows that satisfy a given condition. In this section we will work with a very simple condition, which is that the value in a specified column must be equal to a value that we also specify. The condition is the expression `pl.col('Flavor') == 'chocolate'`.

The code in the cell below creates a DataFrame consisting only of the rows corresponding to chocolate cones.

In [11]:
cones.filter(pl.col('Flavor') == 'chocolate')

Flavor,Color,Price
str,str,f64
"""chocolate""","""light brown""",4.75
"""chocolate""","""dark brown""",5.25
"""chocolate""","""dark brown""",5.25


The expression compares a column with the value we are looking for. The `filter` method can also be used when the condition that the rows must satisfy is more complicated. In those situations the call will be a little more complicated as well.

It is important to provide the value exactly. For example, if we specify `Chocolate` instead of `chocolate`, then `filter` correctly finds no rows where the flavor is `Chocolate`.

In [12]:
cones.filter(pl.col('Flavor') == 'Chocolate')

Flavor,Color,Price
str,str,f64


Like all the other DataFrame methods in this section, `filter` leaves the original DataFrame unchanged.

## Example: Salaries in the NBA

"The NBA is the highest paying professional sports league in the world," [reported CNN](http://edition.cnn.com/2015/12/04/sport/gallery/highest-paid-nba-players/) in March 2016. The DataFrame `nba` contains the [salaries of all National Basketball Association players](https://www.statcrunch.com/app/index.php?dataid=1843341) in 2015-2016.

Each row represents one player. The columns are:

| **Column Label**   | Description                                         |
|--------------------|-----------------------------------------------------|
| `PLAYER`           | Player's name                                       |
| `POSITION`         | Player's position on team                           |
| `TEAM`             | Team name                                           |
|`SALARY`    | Player's salary in 2015-2016, in millions of dollars|
 
The code for the positions is PG (Point Guard), SG (Shooting Guard), PF (Power Forward), SF (Small Forward), and C (Center). But what follows doesn't involve details about how basketball is played.

The first row shows that Paul Millsap, Power Forward for the Atlanta Hawks, had a salary of almost $\$18.7$ million in 2015-2016.

In [13]:
nba

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Paul Millsap""","""PF""","""Atlanta Hawks""",18.671659
"""Al Horford""","""C""","""Atlanta Hawks""",12.0
"""Tiago Splitter""","""C""","""Atlanta Hawks""",9.75625
"""Jeff Teague""","""PG""","""Atlanta Hawks""",8.0
"""Kyle Korver""","""SG""","""Atlanta Hawks""",5.746479
…,…,…,…
"""Gary Neal""","""PG""","""Washington Wizards""",2.139
"""DeJuan Blair""","""C""","""Washington Wizards""",2.0
"""Kelly Oubre Jr.""","""SF""","""Washington Wizards""",1.92024


Fans of Stephen Curry can find his row by using `filter`.

In [14]:
nba.filter(pl.col('PLAYER') == 'Stephen Curry')

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Stephen Curry""","""PG""","""Golden State Warriors""",11.370786


We can also create a new DataFrame called `warriors` consisting of just the data for the Golden State Warriors.

In [15]:
warriors = nba.filter(pl.col('TEAM') == 'Golden State Warriors')
warriors

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Klay Thompson""","""SG""","""Golden State Warriors""",15.501
"""Draymond Green""","""PF""","""Golden State Warriors""",14.26087
"""Andrew Bogut""","""C""","""Golden State Warriors""",13.8
"""Andre Iguodala""","""SF""","""Golden State Warriors""",11.710456
"""Stephen Curry""","""PG""","""Golden State Warriors""",11.370786
…,…,…,…
"""Leandro Barbosa""","""SG""","""Golden State Warriors""",2.5
"""Festus Ezeli""","""C""","""Golden State Warriors""",2.008748
"""Brandon Rush""","""SF""","""Golden State Warriors""",1.270964


By default, the display shows the first five and last five rows. `pl.Config(tbl_rows=-1)` makes `display` show every row.

In [16]:
with pl.Config(tbl_rows=-1):
    display(warriors)

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Klay Thompson""","""SG""","""Golden State Warriors""",15.501
"""Draymond Green""","""PF""","""Golden State Warriors""",14.26087
"""Andrew Bogut""","""C""","""Golden State Warriors""",13.8
"""Andre Iguodala""","""SF""","""Golden State Warriors""",11.710456
"""Stephen Curry""","""PG""","""Golden State Warriors""",11.370786
"""Jason Thompson""","""PF""","""Golden State Warriors""",7.008475
"""Shaun Livingston""","""PG""","""Golden State Warriors""",5.543725
"""Harrison Barnes""","""SF""","""Golden State Warriors""",3.873398
"""Marreese Speights""","""C""","""Golden State Warriors""",3.815


The `nba` DataFrame is sorted in alphabetical order of the team names. To see how the players were paid in 2015-2016, it is useful to sort the data by salary. Remember that by default, the sorting is in increasing order.

In [17]:
nba.sort('SALARY')

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Thanasis Antetokounmpo""","""SF""","""New York Knicks""",0.030888
"""Jordan McRae""","""SG""","""Phoenix Suns""",0.049709
"""Cory Jefferson""","""PF""","""Phoenix Suns""",0.049709
"""Elliot Williams""","""SG""","""Memphis Grizzlies""",0.055722
"""Orlando Johnson""","""SG""","""Phoenix Suns""",0.055722
…,…,…,…
"""Dwight Howard""","""C""","""Houston Rockets""",22.359364
"""Carmelo Anthony""","""SF""","""New York Knicks""",22.875
"""LeBron James""","""SF""","""Cleveland Cavaliers""",22.9705


These figures are somewhat difficult to compare as some of these players changed teams during the season and received salaries from more than one team; only the salary from the last team appears in the DataFrame.  

The CNN report is about the other end of the salary scale – the players who were among the highest paid in the world. To identify these players we can sort in descending order of salary and look at the top few rows.

In [18]:
nba.sort('SALARY', descending=True)

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Kobe Bryant""","""SF""","""Los Angeles Lakers""",25.0
"""Joe Johnson""","""SF""","""Brooklyn Nets""",24.894863
"""LeBron James""","""SF""","""Cleveland Cavaliers""",22.9705
"""Carmelo Anthony""","""SF""","""New York Knicks""",22.875
"""Dwight Howard""","""C""","""Houston Rockets""",22.359364
…,…,…,…
"""Orlando Johnson""","""SG""","""Phoenix Suns""",0.055722
"""Phil Pressey""","""PG""","""Phoenix Suns""",0.055722
"""Jordan McRae""","""SG""","""Phoenix Suns""",0.049709


The late Kobe Bryant was the highest earning NBA player in 2015-2016.